# Stage B Winning-Trace Audit

Inspect the Qwen3-VL-235B teacher traces selected by the Stage B judge before building the SFT dataset.

**Inputs** (under `/mnt/data4/shasta/amar.amarjyoti/research_data/vlm_cot_distill/`):
- `_train_qa_for_cot.jsonl` — source QAs keyed by `id`
- `cot_1058163_..._grounding.jsonl` — Stage A traces, keyed by `(id, sample_idx)`
- `judge_1778033257_qwen32b.jsonl` — Stage B judge (canonical 41,079-row run)

**Image-grounded gate (D1)** — handoff §Stage D formatter:
```
parse_ok_b1[best_idx]
AND answer_correctness == 1
AND hallucination       >= 4
AND visual_grounding    >= 4
AND reasoning_quality   >= 3
```

In [ ]:
import json, gzip, os, sys, math, random, re
from collections import Counter, defaultdict
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, Markdown

pd.set_option('display.max_colwidth', 200)
plt.rcParams['figure.dpi'] = 110

DATA = Path('/mnt/data4/shasta/amar.amarjyoti/research_data/vlm_cot_distill')
QA_FILE     = DATA / '_train_qa_for_cot.jsonl'
COT_FILE    = DATA / 'cot_1058163_Qwen3-VL-235B-A22B-Thinking-FP8_train_N16_T0.8_grounding.jsonl'
JUDGE_FILE  = DATA / 'judge_1778033257_qwen32b.jsonl'

# D1 image-grounded gate
GATE = dict(answer_correctness=1, hallucination=4, visual_grounding=4, reasoning_quality=3)

OUT_DIR = Path('.') / 'audit_out'
OUT_DIR.mkdir(exist_ok=True)

for p in [QA_FILE, COT_FILE, JUDGE_FILE]:
    print(f'{p.name:75s} {p.stat().st_size/1e9:7.3f} GB')

## 1. Load judge (streaming, drop `raw_b1`/`raw_b2` to keep RAM small)

In [ ]:
def iter_jsonl(path):
    with open(path) as f:
        for ln in f:
            ln = ln.strip()
            if ln:
                yield json.loads(ln)

judge_by_id = {}
DROP = {'raw_b1', 'raw_b2'}
for r in iter_jsonl(JUDGE_FILE):
    judge_by_id[r['id']] = {k: v for k, v in r.items() if k not in DROP}
print(f'judge records: {len(judge_by_id):,}')
print('example keys:', list(next(iter(judge_by_id.values())).keys()))

## 2. Load QA metadata

In [ ]:
qa_by_id = {r['id']: r for r in iter_jsonl(QA_FILE)}
print(f'QA records: {len(qa_by_id):,}')

## 3. Stream Stage A; keep only winning `(id, best_idx)`

In [ ]:
wanted = {(rid, j['best_idx']) for rid, j in judge_by_id.items() if j.get('best_idx') is not None}
print(f'winning (id, best_idx) pairs: {len(wanted):,}')

winners = {}
for r in iter_jsonl(COT_FILE):
    key = (r['id'], r['sample_idx'])
    if key in wanted:
        winners[key] = {
            'grounding'        : r.get('grounding') or '',
            'thinking'         : r.get('thinking') or '',
            'answer'           : r.get('answer') or '',
            'finish_reason'    : r.get('finish_reason'),
            'num_output_tokens': r.get('num_output_tokens'),
        }
print(f'matched winners: {len(winners):,}')

## 4. Build per-`id` audit dataframe

In [ ]:
OBJ_RE = re.compile(r'<obj\d+>')
WORD_RE = re.compile(r"\S+")

def wc(s):
    return len(WORD_RE.findall(s or ''))

rows = []
for rid, j in judge_by_id.items():
    qa = qa_by_id.get(rid, {})
    best = j.get('best_idx')
    w = winners.get((rid, best), {}) if best is not None else {}
    scores = j.get('scores') or []
    best_score = scores[best] if (best is not None and best < len(scores) and scores[best]) else None
    pob = (j.get('parse_ok_b1') or [])
    parse_ok_best = pob[best] if (best is not None and best < len(pob)) else False

    thinking = w.get('thinking', '')
    answer   = w.get('answer', '')
    grounding = w.get('grounding', '')

    rows.append(dict(
        id=rid,
        task_family=qa.get('task_family'),
        template_type=qa.get('template_type'),
        image_path=qa.get('image_path'),
        gt_answer=qa.get('gt_answer'),
        best_idx=best,
        all_b1_failed=j.get('all_b1_failed', False),
        parse_ok_best=bool(parse_ok_best),
        oversized_b1=bool(j.get('oversized_b1', False)),
        b1_chat_error=j.get('b1_chat_error') or None,
        score_total=(best_score or {}).get('score_total'),
        ans_correct=(best_score or {}).get('answer_correctness'),
        hallu=((best_score or {}).get('hallucination') or {}).get('score'),
        ground=((best_score or {}).get('visual_grounding') or {}).get('score'),
        reason=((best_score or {}).get('reasoning_quality') or {}).get('score'),
        scene_desc_words=wc(j.get('scene_description')),
        thinking_words=wc(thinking),
        thinking_chars=len(thinking),
        answer_words=wc(answer),
        answer_chars=len(answer),
        grounding_objs=len(OBJ_RE.findall(grounding)),
        grounding_chars=len(grounding),
        num_output_tokens=w.get('num_output_tokens'),
        finish_reason=w.get('finish_reason'),
    ))
df = pd.DataFrame(rows)
print(df.shape)
df.head(3)

## 4b. Fix `answer_correctness` for continuous templates (xy2d, depth)

The judge scored `answer_correctness` by near-exact string match. Two `template_type`s have
**continuous** gold answers and so score ~0% under exact match — they are then dropped by the D1
gate even when the teacher is essentially right:

| template | gold example | predicted | correct metric |
|---|---|---|---|
| `xy2d`  | `[385, 564]` (pixels) | `[237, 614]` (Qwen **0–1000 normalised**) | Euclidean distance after rescaling pred by `(W,H)/1000` |
| `depth` | `Between 21 meters and 28 meters` | `Between 15 meters and 19 meters` | `|midpoint(pred) − midpoint(gold)|` (metres) |

**Why xy2d looked broken:** the teacher emits coordinates in Qwen's 0–1000 space while gold is in
absolute pixels (1600×900). Raw distance is ~330 px (median); after rescaling it drops to ~20 px.

This cell recomputes `answer_correctness` for every candidate of these two templates from an
**absolute difference** + a per-template tolerance (`CONT_TOL`, tunable), **re-selects the winning
trace** to a correct+high-quality candidate, and **rebuilds `df`**. Everything downstream
(correctness rate §5b, D1 gate §6, per-family tables, exports) then uses the corrected values.
Categorical templates (`distance`, `lr`, `fb`, `yaw`) are left exactly as the judge scored them.

In [ ]:
# ===== Recompute answer_correctness for CONTINUOUS templates =====
# The Stage-B judge graded answer_correctness by (near-)exact string match. That is wrong for
# templates whose gold answer is a continuous quantity: xy2d (a 2-D point) and depth (a metric
# range). For those the judge returns ~0 everywhere, so they are silently excluded by the D1 gate.
#
# Fix: grade by an ABSOLUTE DIFFERENCE, then threshold with a per-template tolerance (tune freely).
#   * xy2d  -> Euclidean distance (pixels).  NOTE: the teacher emits Qwen 0-1000 NORMALISED coords,
#              while gold is in absolute pixels, so pred is rescaled by (W,H)/1000 first.
#   * depth -> |midpoint(pred_range) - midpoint(gold_range)|  (metres).
# Categorical templates (distance, lr, fb, yaw) keep the judge's answer_correctness untouched.

CONT_TEMPLATES = {'xy2d', 'depth'}
CONT_TOL = {'xy2d': 50.0, 'depth': 4.0}        # xy2d: pixels | depth: metres  <-- tune here
_NUM = re.compile(r'-?\d+(?:\.\d+)?')

def _img_size(path, _cache={}):
    if path in _cache:
        return _cache[path]
    try:
        with Image.open(path) as im:
            wh = im.size                        # lazy: reads header only
    except Exception:
        wh = (1600, 900)                        # nuScenes default if unreadable
    _cache[path] = wh
    return wh

def _parse_point(s):
    n = _NUM.findall(s or '')
    return (float(n[0]), float(n[1])) if len(n) >= 2 else None

def _parse_depth_mid(s):
    n = [float(x) for x in _NUM.findall(s or '')]
    if not n:
        return None
    return (n[0] + n[1]) / 2.0 if len(n) >= 2 else n[0]

def cont_error(template, pred_ans, gold_ans, image_path):
    """Absolute difference for a continuous answer; None if unparseable."""
    if template == 'xy2d':
        p, g = _parse_point(pred_ans), _parse_point(gold_ans)
        if not p or not g:
            return None
        W, H = _img_size(image_path)
        p = (p[0] * W / 1000.0, p[1] * H / 1000.0)   # 0-1000 normalised -> pixels
        return math.hypot(p[0] - g[0], p[1] - g[1])
    if template == 'depth':
        p, g = _parse_depth_mid(pred_ans), _parse_depth_mid(gold_ans)
        if p is None or g is None:
            return None
        return abs(p - g)
    return None

cont_ids = {rid for rid, qa in qa_by_id.items() if qa.get('template_type') in CONT_TEMPLATES}
print(f'continuous ids (xy2d+depth): {len(cont_ids):,}')

# Stream Stage A once, keeping ALL 16 candidates for continuous ids.
cont_cand = defaultdict(dict)      # id -> {sample_idx: winner-style record}
cont_err  = defaultdict(dict)      # id -> {sample_idx: error or None}
for r in iter_jsonl(COT_FILE):
    rid = r['id']
    if rid not in cont_ids:
        continue
    qa = qa_by_id[rid]
    idx = r['sample_idx']
    cont_cand[rid][idx] = {
        'grounding'        : r.get('grounding') or '',
        'thinking'         : r.get('thinking') or '',
        'answer'           : r.get('answer') or '',
        'finish_reason'    : r.get('finish_reason'),
        'num_output_tokens': r.get('num_output_tokens'),
    }
    cont_err[rid][idx] = cont_error(qa['template_type'], r.get('answer'),
                                    qa.get('gt_answer'), qa.get('image_path'))
print(f'collected candidates for {len(cont_cand):,} continuous ids')

# Overwrite judge answer_correctness per candidate, then re-select the winning trace.
n_reselected = 0
for rid in cont_ids:
    j = judge_by_id.get(rid)
    if not j:
        continue
    tmpl = qa_by_id[rid]['template_type']
    tol  = CONT_TOL[tmpl]
    scores = j.get('scores') or []
    pob    = j.get('parse_ok_b1') or []
    errs   = cont_err.get(rid, {})

    for idx, s in enumerate(scores):
        if not s:
            continue
        e = errs.get(idx)
        s['cont_error'] = e
        s['answer_correctness'] = int(e is not None and e <= tol)

    # candidates that (a) the judge parsed and (b) have a parseable predicted answer
    cand = [(idx, errs[idx], (scores[idx] or {}).get('score_total') or 0)
            for idx in errs
            if errs[idx] is not None and idx < len(scores) and scores[idx]
            and idx < len(pob) and pob[idx]]
    correct = [c for c in cand if c[1] <= tol]
    if correct:                                   # prefer a correct trace, best judge composite
        best_idx = max(correct, key=lambda c: (c[2], -c[1]))[0]
    elif cand:                                    # else the spatially closest parseable trace
        best_idx = min(cand, key=lambda c: c[1])[0]
    else:
        best_idx = j.get('best_idx')              # nothing usable: leave as-is (will fail gate)

    if best_idx is not None and best_idx != j.get('best_idx'):
        n_reselected += 1
    if best_idx is not None:
        j['best_idx'] = best_idx
        rec = cont_cand[rid].get(best_idx)
        if rec is not None:
            winners[(rid, best_idx)] = rec
print(f're-selected winning trace for {n_reselected:,} continuous ids')
del cont_cand                                     # free memory

# ---- Rebuild df from the corrected judge_by_id / winners (same logic as the cell above) ----
rows = []
for rid, j in judge_by_id.items():
    qa = qa_by_id.get(rid, {})
    best = j.get('best_idx')
    w = winners.get((rid, best), {}) if best is not None else {}
    scores = j.get('scores') or []
    best_score = scores[best] if (best is not None and best < len(scores) and scores[best]) else None
    pob = (j.get('parse_ok_b1') or [])
    parse_ok_best = pob[best] if (best is not None and best < len(pob)) else False
    tmpl = qa.get('template_type')

    thinking = w.get('thinking', ''); answer = w.get('answer', ''); grounding = w.get('grounding', '')
    rows.append(dict(
        id=rid, task_family=qa.get('task_family'), template_type=tmpl,
        image_path=qa.get('image_path'), gt_answer=qa.get('gt_answer'),
        best_idx=best, all_b1_failed=j.get('all_b1_failed', False),
        parse_ok_best=bool(parse_ok_best), oversized_b1=bool(j.get('oversized_b1', False)),
        b1_chat_error=j.get('b1_chat_error') or None,
        score_total=(best_score or {}).get('score_total'),
        ans_correct=(best_score or {}).get('answer_correctness'),
        hallu=((best_score or {}).get('hallucination') or {}).get('score'),
        ground=((best_score or {}).get('visual_grounding') or {}).get('score'),
        reason=((best_score or {}).get('reasoning_quality') or {}).get('score'),
        scene_desc_words=wc(j.get('scene_description')),
        thinking_words=wc(thinking), thinking_chars=len(thinking),
        answer_words=wc(answer), answer_chars=len(answer),
        grounding_objs=len(OBJ_RE.findall(grounding)), grounding_chars=len(grounding),
        num_output_tokens=w.get('num_output_tokens'), finish_reason=w.get('finish_reason'),
        is_continuous=tmpl in CONT_TEMPLATES,
        cont_error=(best_score or {}).get('cont_error'),
    ))
df = pd.DataFrame(rows)

# ---- Before/after summary for the two continuous templates ----
for t in sorted(CONT_TEMPLATES):
    sub = df[df.template_type == t]
    e = pd.Series([cont_err[i].get(judge_by_id[i].get('best_idx'))
                   for i in sub.id if i in cont_err]).dropna()
    rate = (sub.ans_correct == 1).mean() * 100
    print(f'\n{t:6s}: winner answer_correctness now {rate:5.1f}%  (was ~0% under exact-match)')
    if len(e):
        for tol in ([20, 30, 50, 100] if t == 'xy2d' else [1, 2, 4, 8]):
            print(f'        tol<={tol:>4}{"px" if t=="xy2d" else "m"}: winner correct = '
                  f'{(e <= tol).mean()*100:5.1f}%')
print(f'\ndf rebuilt: {df.shape}')
df.head(3)

## 5. Coverage & failure-mode counts

In [ ]:
N = len(df)
checks = {
    'total judged'           : N,
    'all_b1_failed'          : int(df.all_b1_failed.sum()),
    'b1_chat_error'          : int(df.b1_chat_error.notna().sum()),
    'oversized_b1'           : int(df.oversized_b1.sum()),
    'best_idx missing'       : int(df.best_idx.isna().sum()),
    'parse_ok_best == False' : int((~df.parse_ok_best).sum()),
    'winner trace missing'   : int(df.thinking_chars.eq(0).sum()),
}
pd.Series(checks).to_frame('count').assign(pct=lambda x: 100*x['count']/N)

## 5b. B1 vs B2 — per-stage health and comparison

Stage B runs two judge sub-stages:

- **B1** — score each of the 16 candidate traces individually (image + Q + gold + trace_i). 16 calls per id; produces `scores[]`, `parse_ok_b1[]`, `oversized_b1[]`, `raw_b1[]`.
- **B2** — given the winning trace only, write a text-only scene description for downstream Stage C polish. 1 call per id; produces `scene_description`, `parse_ok_b2`, `oversized_b2`, `raw_b2`.

This section audits each stage independently and compares failure overlap, lengths, and effective cost.

In [ ]:
# ---- B1 per-sample aggregation (16 entries per id) + B2 per-id ----
b1_total = b1_parse_ok = b1_oversize = b1_chat_err_ids = all_b1_failed_n = 0
b2_total = b2_parse_ok = b2_oversize = b2_chat_err = 0
b1_per_id_pass = []
b2_scene_words = []

for rid, j in judge_by_id.items():
    pob = j.get('parse_ok_b1') or []
    ovb = j.get('oversized_b1') or []
    n_pass = sum(1 for x in pob if x)
    b1_per_id_pass.append(n_pass)
    b1_total     += len(pob)
    b1_parse_ok  += n_pass
    b1_oversize  += sum(1 for x in ovb if x)
    if j.get('b1_chat_error'):  b1_chat_err_ids += 1
    if j.get('all_b1_failed'):  all_b1_failed_n += 1

    b2_total += 1
    if j.get('parse_ok_b2'):    b2_parse_ok += 1
    if j.get('oversized_b2'):   b2_oversize += 1
    if j.get('b2_chat_error'):  b2_chat_err += 1
    b2_scene_words.append(len(WORD_RE.findall(j.get('scene_description') or '')))

b1_per_id = np.asarray(b1_per_id_pass)
n_ids = len(judge_by_id)

summary_b = pd.DataFrame([
    dict(stage='B1 (score, 16/id)', calls_per_id=16, total_calls=b1_total,
         parse_ok_rate=b1_parse_ok/max(1,b1_total),
         oversize_rate=b1_oversize/max(1,b1_total),
         chat_error_rate_id=b1_chat_err_ids/max(1,n_ids)),
    dict(stage='B2 (describe, 1/id)', calls_per_id=1, total_calls=b2_total,
         parse_ok_rate=b2_parse_ok/max(1,b2_total),
         oversize_rate=b2_oversize/max(1,b2_total),
         chat_error_rate_id=b2_chat_err/max(1,b2_total)),
])
display(summary_b.style.format({'parse_ok_rate':'{:.4f}','oversize_rate':'{:.4f}','chat_error_rate_id':'{:.4f}'}))

print(f'ids with all_b1_failed: {all_b1_failed_n:,} ({100*all_b1_failed_n/n_ids:.2f}%)')
print(f'B1 mean parsed-per-id : {b1_per_id.mean():.2f} / 16  (median {int(np.median(b1_per_id))})')
print(f'B2 scene_description : mean {np.mean(b2_scene_words):.0f} words, median {int(np.median(b2_scene_words))}')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

vc = np.bincount(b1_per_id, minlength=17)
axes[0].bar(range(17), vc)
axes[0].set_xlabel('# of 16 B1 samples that parsed'); axes[0].set_ylabel('# ids')
axes[0].set_title(f'B1 parse success per id  (median {int(np.median(b1_per_id))} / 16)')

axes[1].hist(b2_scene_words, bins=40, edgecolor='black')
axes[1].set_xlabel('scene_description words'); axes[1].set_ylabel('# ids')
axes[1].set_title(f'B2 scene_description length  (mean {np.mean(b2_scene_words):.0f}w)')

b1_fail = np.array([bool(j.get('all_b1_failed')) for j in judge_by_id.values()])
b2_fail = np.array([not bool(j.get('parse_ok_b2')) for j in judge_by_id.values()])
ct = pd.crosstab(pd.Series(b1_fail, name='B1 all-failed'),
                 pd.Series(b2_fail, name='B2 parse-fail'))
ct = ct.reindex(index=[False,True], columns=[False,True], fill_value=0)
axes[2].imshow(ct.values, cmap='Reds')
axes[2].set_xticks([0,1]); axes[2].set_xticklabels(['B2 OK','B2 fail'])
axes[2].set_yticks([0,1]); axes[2].set_yticklabels(['B1 OK','B1 all-fail'])
for i in range(2):
    for k in range(2):
        axes[2].text(k, i, f'{ct.values[i,k]:,}', ha='center', va='center', fontsize=11)
axes[2].set_title('B1 vs B2 failure cross-tab (per id)')

plt.tight_layout(); plt.savefig(OUT_DIR / 'b1_vs_b2_health.png'); plt.show()

In [ ]:
# ---- Per-id correctness rate: #correct traces / 16 ----
correct_per_id, parsed_per_id = [], []
for rid in df.id:
    scores = (judge_by_id.get(rid) or {}).get('scores') or []
    n_correct = sum(1 for s in scores if s and s.get('answer_correctness') == 1)
    n_parsed  = sum(1 for s in scores if s)
    correct_per_id.append(n_correct)
    parsed_per_id.append(n_parsed)

df['n_correct_16']   = correct_per_id
df['n_parsed_16']    = parsed_per_id
df['correct_rate_16']    = df.n_correct_16 / 16
df['correct_rate_parsed'] = np.where(df.n_parsed_16 > 0, df.n_correct_16 / df.n_parsed_16.clip(lower=1), np.nan)

n0  = int((df.n_correct_16 == 0).sum())
n16 = int((df.n_correct_16 == 16).sum())
print(f'mean correctness /16        : {df.correct_rate_16.mean():.3f}  (median {df.correct_rate_16.median():.3f})')
print(f'mean correctness /parsed    : {df.correct_rate_parsed.mean():.3f}')
print(f'ids with 0/16 correct       : {n0:,} ({100*n0/len(df):.2f}%)')
print(f'ids with 16/16 correct      : {n16:,} ({100*n16/len(df):.2f}%)')
print(f'ids in GRPO sweet spot 25-75%: {int(df.correct_rate_16.between(.25,.75).sum()):,}')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

vc = np.bincount(df.n_correct_16.values, minlength=17)
axes[0].bar(range(17), vc)
axes[0].set_xlabel('# correct traces (of 16)'); axes[0].set_ylabel('# ids')
axes[0].set_title(f'Per-id correct-trace count  (mean {df.correct_rate_16.mean()*16:.1f}/16)')

fam_c = df.groupby('task_family').correct_rate_16.agg(['size','mean']).sort_values('mean')
fam_c.columns = ['n','mean_rate']
axes[1].barh(fam_c.index, fam_c.mean_rate)
for i, (n, r) in enumerate(zip(fam_c.n, fam_c.mean_rate)):
    axes[1].text(r, i, f'  {r*100:.1f}% (n={n})', va='center', fontsize=8)
axes[1].set_xlim(0, 1); axes[1].set_xlabel('mean correctness rate /16')
axes[1].set_title('Per-task_family mean correctness rate')
plt.tight_layout(); plt.savefig(OUT_DIR / 'correctness_rate.png'); plt.show()

fam_c

## 5b-bis. Answer-correctness per `template_type` subtask

The §5b chart groups by `task_family`, but every id here is the single family
`spatial_relation_geometry`, so the real per-subtask signal lives in `template_type`
(`lr`, `fb`, `yaw`, `distance`, `xy2d`, `depth`). This breaks **winner answer_correctness**
and **per-id correctness-rate /16** down by subtask, with the two continuous templates
(`xy2d`, `depth`) now scored by the absolute-difference metric from §4b instead of exact match.

In [ ]:
# ---- Answer-correctness per template_type subtask ----
# winner correctness = best trace lands on gold (post-4b fix for xy2d/depth)
# rate_/16          = mean #correct of the 16 candidates per id
sub = (df.groupby('template_type')
         .agg(n=('id', 'size'),
              winner_correct=('ans_correct', lambda s: (s == 1).mean()),
              rate_16=('correct_rate_16', 'mean'),
              is_continuous=('is_continuous', 'first'))
         .sort_values('winner_correct'))
display(sub.assign(winner_correct=(sub.winner_correct*100).round(1),
                   rate_16=(sub.rate_16*100).round(1)))

fig, ax = plt.subplots(figsize=(8, 4))
y = np.arange(len(sub)); w = 0.4
ax.barh(y - w/2, sub.winner_correct*100, w, label='winner correct',
        color=['tab:orange' if c else 'tab:blue' for c in sub.is_continuous])
ax.barh(y + w/2, sub.rate_16*100, w, label='mean correct /16', color='lightgray')
for i, (wc_, r_, n_) in enumerate(zip(sub.winner_correct, sub.rate_16, sub.n)):
    ax.text(wc_*100, i - w/2, f' {wc_*100:.0f}% (n={n_})', va='center', fontsize=8)
ax.set_yticks(y); ax.set_yticklabels(sub.index)
ax.set_xlim(0, 100); ax.set_xlabel('%'); ax.legend(fontsize=8)
ax.set_title('Answer-correctness per template_type subtask (orange = continuous, §4b metric)')
plt.tight_layout(); plt.savefig(OUT_DIR / 'correctness_by_template.png'); plt.show()


## 5c. B1 (teacher per-sample) vs Stage C (DeepSeek text-only polish)

For each id we have two "did this reasoning land on gold?" signals:

- **B1 winner** — teacher's best of 16 image-conditioned traces (`ans_correct == 1`).
- **Stage C** — DeepSeek text-only re-derivation from B2's `scene_description` (`c2_lands_on_gold`).

Useful comparisons:

1. **Correctness agreement** — does the text-only polish recover gold whenever the teacher does, and where does it diverge?
2. **Reasoning length** — does C reason as long as the teacher? Headline ratio for "is the scene_description carrying enough signal".
3. **Failure modes** — meta-leak ("the description says…"), reflective-marker rate, parse/oversize/chat-error rates.
4. **Per task_family** — which families does the text-only polish fail on (likely those that need direct visual grounding).

Defaults to the largest Stage C output (1060224, 38,848 records, non-thinking polish). Swap `PHASE_C_FILE` for the v3_thinking run if you want to compare variants.

In [ ]:
# Stage C = DeepSeek with THINKING ON (v3). 35,040 recs, 96.1% carry a c2_think trace
# (mean ~616 words), lands_on_gold 79.7%. This is the run we actually want to audit.
# (The older 1060224 file is the non-thinking polish -> empty c2_think; don't use it here.)
PHASE_C_FILE = DATA / 'phase_c_1060281_deepseekv4_c2_v3_thinking.jsonl'

META_LEAK_RE = re.compile(
    r'\b(the (scene\s+)?description|the (image|picture|photo)|in the image|'
    r'as (shown|noted|stated|described|mentioned)|according to (the )?(scene\s+)?description|'
    r'based on the (text|description)|prior|previous|given (trace|reasoning))\b',
    re.IGNORECASE)
REFLECT_RE = re.compile(
    r'\b(wait|but actually|hmm|let me (check|reconsider|re-?evaluate|verify|think again)|'
    r'on second thought|reconsider|actually,|on closer|however|alternatively|'
    r'could (also|instead)|that would mean|let me re-?check)\b',
    re.IGNORECASE)

phase_c = {r['id']: r for r in iter_jsonl(PHASE_C_FILE)}
print(f'Stage C records: {len(phase_c):,} (from {PHASE_C_FILE.name})')

df_idx = df.set_index('id')
rows = []
for rid, c in phase_c.items():
    if rid not in df_idx.index:
        continue
    d = df_idx.loc[rid]
    ct = c.get('c2_think') or ''
    ca = c.get('c2_answer') or ''
    rows.append(dict(
        id=rid,
        task_family=d.task_family,
        b1_score_total=d.score_total,
        b1_correct=int(d.ans_correct == 1) if pd.notna(d.ans_correct) else 0,
        b1_thinking_words=d.thinking_words,
        c_lands=bool(c.get('c2_lands_on_gold')),
        c_parse_ok=bool(c.get('parse_ok_c2')),
        c_oversize=bool(c.get('oversized_c2')),
        c_chat_err=bool(c.get('c2_chat_error')),
        c_think_words=len(WORD_RE.findall(ct)),
        c_answer_words=len(WORD_RE.findall(ca)),
        c_meta_leak=bool(META_LEAK_RE.search(ct)),
        c_reflective=bool(REFLECT_RE.search(ct)),
    ))
cmp_df = pd.DataFrame(rows)
print(f'matched B1↔C pairs: {len(cmp_df):,}')

print()
print(f'B1 winner correct  : {cmp_df.b1_correct.mean()*100:.2f}%')
print(f'C lands_on_gold    : {cmp_df.c_lands.mean()*100:.2f}%')
print(f'C parse_ok         : {cmp_df.c_parse_ok.mean()*100:.2f}%')
print(f'C oversized        : {cmp_df.c_oversize.mean()*100:.2f}%')
print(f'C meta-leak        : {cmp_df.c_meta_leak.mean()*100:.2f}%')
print(f'C reflective       : {cmp_df.c_reflective.mean()*100:.2f}%')
print(f'mean think words   : B1={cmp_df.b1_thinking_words.mean():.0f}  C={cmp_df.c_think_words.mean():.0f}  '
      f'ratio C/B1 = {cmp_df.c_think_words.mean()/max(1,cmp_df.b1_thinking_words.mean()):.2f}')

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# 1. Correctness cross-tab
ct = pd.crosstab(cmp_df.b1_correct == 1, cmp_df.c_lands)
ct = ct.reindex(index=[False,True], columns=[False,True], fill_value=0)
axes[0,0].imshow(ct.values, cmap='Blues')
axes[0,0].set_xticks([0,1]); axes[0,0].set_xticklabels(['C miss','C lands'])
axes[0,0].set_yticks([0,1]); axes[0,0].set_yticklabels(['B1 wrong','B1 correct'])
for i in range(2):
    for k in range(2):
        axes[0,0].text(k, i, f'{ct.values[i,k]:,}', ha='center', va='center', fontsize=11)
axes[0,0].set_title('B1 winner correct  vs  C lands_on_gold')

# 2 & 3. Reasoning length overlay + per-id scatter.
#   The non-thinking Stage C polish (e.g. 1060224) has EMPTY c2_think, so a thinking-vs-thinking
#   comparison selects nothing. Fall back to c2_answer length in that case, and guard the empty set
#   so the cell never crashes (np.percentile on an empty array raises IndexError).
has_think = cmp_df.c_think_words.gt(0).any()
c_col, c_lab = ('c_think_words', 'C c2_think') if has_think else ('c_answer_words', 'C c2_answer (no thinking trace)')
if not has_think:
    print('NOTE: selected Stage C file has no c2_think (non-thinking polish) -> '
          'length panels compare B1 thinking vs C ANSWER words. '
          'Use a *_v3_thinking phase_c file for a thinking-vs-thinking comparison.')
mask = (cmp_df.b1_thinking_words > 0) & (cmp_df[c_col] > 0)
both = cmp_df[mask]
if len(both):
    hi = float(np.percentile(np.r_[both.b1_thinking_words, both[c_col]], 99))
    bins = np.linspace(0, max(hi, 1), 50)
    axes[0,1].hist(both.b1_thinking_words, bins=bins, alpha=0.55,
                   label=f'B1 thinking (mean {both.b1_thinking_words.mean():.0f}w)', color='tab:blue')
    axes[0,1].hist(both[c_col], bins=bins, alpha=0.55,
                   label=f'{c_lab} (mean {both[c_col].mean():.0f}w)', color='tab:orange')
    axes[0,1].legend(fontsize=9)
    sub = both.sample(min(8000, len(both)), random_state=0)
    lim = float(max(both.b1_thinking_words.quantile(.99), both[c_col].quantile(.99), 1))
    axes[1,0].scatter(sub.b1_thinking_words, sub[c_col], s=3, alpha=0.25)
    axes[1,0].plot([0, lim], [0, lim], 'k--', lw=0.5)
    axes[1,0].set_xlim(0, lim); axes[1,0].set_ylim(0, lim)
else:
    for _ax in (axes[0,1], axes[1,0]):
        _ax.text(0.5, 0.5, 'no comparable length data', ha='center', va='center', transform=_ax.transAxes)
axes[0,1].set_xlabel('words'); axes[0,1].set_title('Reasoning length: B1 vs C')
axes[1,0].set_xlabel('B1 thinking words'); axes[1,0].set_ylabel(f'{c_lab} words')
axes[1,0].set_title('Per-id reasoning length: B1 vs C')

# 4. Per task_family lands-rate
fam_cmp = cmp_df.groupby('task_family').agg(
    n=('id','size'),
    b1=('b1_correct','mean'),
    c =('c_lands','mean'),
).sort_values('b1')
y = np.arange(len(fam_cmp)); w = 0.4
axes[1,1].barh(y - w/2, fam_cmp.b1, w, label='B1 winner correct', color='tab:blue')
axes[1,1].barh(y + w/2, fam_cmp.c,  w, label='C lands_on_gold',  color='tab:orange')
axes[1,1].set_yticks(y); axes[1,1].set_yticklabels(fam_cmp.index, fontsize=8)
axes[1,1].set_xlim(0, 1); axes[1,1].legend(fontsize=8)
axes[1,1].set_title('Per task_family: B1 vs C correctness')

plt.tight_layout(); plt.savefig(OUT_DIR / 'b1_vs_c.png'); plt.show()
fam_cmp.assign(gap=fam_cmp.b1 - fam_cmp.c).sort_values('gap', ascending=False)

In [ ]:
# Re-stream judge file once to compare raw_b1 vs raw_b2 char-length distributions.
# (~10 GB, takes a few minutes. Skip this cell if you only need the health metrics above.)
b1_raw_winner = []
b1_raw_all    = []
b2_raw        = []
n_scanned = 0
for r in iter_jsonl(JUDGE_FILE):
    raws = r.get('raw_b1') or []
    if raws:
        best = r.get('best_idx')
        if best is not None and 0 <= best < len(raws):
            b1_raw_winner.append(len(raws[best] or ''))
        b1_raw_all.extend(len(s or '') for s in raws)
    b2_raw.append(len(r.get('raw_b2') or ''))
    n_scanned += 1
print(f'scanned {n_scanned:,} judge records')

fig, ax = plt.subplots(figsize=(9, 4))
hi = float(np.percentile(b1_raw_winner + b2_raw, 99))
bins = np.linspace(0, hi, 60)
ax.hist(b1_raw_winner, bins=bins, alpha=0.55, label=f'raw_b1[best] (n={len(b1_raw_winner):,})', color='tab:blue')
ax.hist(b2_raw,        bins=bins, alpha=0.55, label=f'raw_b2 (n={len(b2_raw):,})',               color='tab:orange')
ax.set_xlabel('chars'); ax.set_title('B1 (winning sample) vs B2 raw-output char length')
ax.legend(); plt.tight_layout(); plt.savefig(OUT_DIR / 'b1_b2_raw_len.png'); plt.show()

# Cost comparison: total tokens (or chars) issued for B1 (×16) vs B2 (×1)
total_b1_chars = sum(b1_raw_all)
total_b2_chars = sum(b2_raw)
print(f'B1 total raw chars: {total_b1_chars/1e9:.2f} GB    (16× per id)')
print(f'B2 total raw chars: {total_b2_chars/1e9:.2f} GB    (1× per id)')
print(f'cost ratio B1/B2:   {total_b1_chars/max(1,total_b2_chars):.2f}x')

pd.DataFrame({
    'b1_raw_winner': pd.Series(b1_raw_winner).describe(percentiles=[.5, .9, .95, .99]),
    'b1_raw_all':    pd.Series(b1_raw_all).describe(percentiles=[.5, .9, .95, .99]),
    'b2_raw':        pd.Series(b2_raw).describe(percentiles=[.5, .9, .95, .99]),
}).round(0)

## 6. D1 image-grounded gate funnel

In [ ]:
conds = [
    ('parse_ok_best',           df.parse_ok_best),
    ('ans_correct == 1',        df.ans_correct == GATE['answer_correctness']),
    (f'hallu >= {GATE["hallucination"]}',         df.hallu  >= GATE['hallucination']),
    (f'ground >= {GATE["visual_grounding"]}',     df.ground >= GATE['visual_grounding']),
    (f'reason >= {GATE["reasoning_quality"]}',    df.reason >= GATE['reasoning_quality']),
]
mask = pd.Series(True, index=df.index)
funnel = []
for name, c in conds:
    c = c.fillna(False)
    mask &= c
    funnel.append((name, int(c.sum()), int(mask.sum())))
funnel_df = pd.DataFrame(funnel, columns=['condition', 'pass_indep', 'pass_cumulative'])
funnel_df['cum_pct'] = 100*funnel_df.pass_cumulative/N
df['pass_gate'] = mask
print(f'Final gated dataset: {int(mask.sum()):,} / {N:,} ({100*mask.mean():.1f}%)')
funnel_df

In [ ]:
# Gate pass rate per task_family / template_type
fam = df.groupby('task_family')['pass_gate'].agg(['size','sum','mean']).sort_values('size', ascending=False)
fam.columns = ['n','passed','pass_rate']
display(fam)

tmpl = df.groupby('template_type')['pass_gate'].agg(['size','sum','mean']).sort_values('size', ascending=False)
tmpl.columns = ['n','passed','pass_rate']
tmpl.head(20)

## 7. Judge-score distributions

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for ax, col, title in zip(
    axes.flat,
    ['score_total', 'hallu', 'ground', 'reason', 'ans_correct'],
    ['best score_total', 'hallucination', 'visual_grounding', 'reasoning_quality', 'answer_correctness'],
):
    s = df[col].dropna()
    ax.hist(s, bins=20, edgecolor='black')
    ax.set_title(f'{title}  (mean={s.mean():.2f})')
axes.flat[-1].axis('off')
plt.tight_layout()
plt.savefig(OUT_DIR / 'judge_scores.png'); plt.show()

In [ ]:
# Axis correlation
corr_cols = ['score_total','hallu','ground','reason','ans_correct','thinking_words','grounding_objs']
corr = df[corr_cols].corr()
fig, ax = plt.subplots(figsize=(6,5))
im = ax.imshow(corr, vmin=-1, vmax=1, cmap='RdBu_r')
ax.set_xticks(range(len(corr_cols))); ax.set_xticklabels(corr_cols, rotation=45, ha='right')
ax.set_yticks(range(len(corr_cols))); ax.set_yticklabels(corr_cols)
for i in range(len(corr_cols)):
    for k in range(len(corr_cols)):
        ax.text(k, i, f'{corr.iloc[i,k]:.2f}', ha='center', va='center', fontsize=8)
plt.colorbar(im, ax=ax, shrink=0.7); plt.title('score axis correlations'); plt.tight_layout()
plt.savefig(OUT_DIR / 'score_corr.png'); plt.show()

## 8. Length distributions (passed gate vs filtered)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 7))
len_cols = [('thinking_words','reasoning words'),
            ('answer_words','answer words'),
            ('scene_desc_words','scene_description words'),
            ('num_output_tokens','teacher output tokens')]
for ax, (col, title) in zip(axes.flat, len_cols):
    a = df.loc[ df.pass_gate, col].dropna()
    b = df.loc[~df.pass_gate, col].dropna()
    hi = np.nanpercentile(pd.concat([a,b]), 99) if len(a)+len(b) else 1
    bins = np.linspace(0, hi, 50)
    ax.hist(b, bins=bins, alpha=0.5, label=f'filtered (n={len(b):,})', color='tab:red')
    ax.hist(a, bins=bins, alpha=0.6, label=f'passed (n={len(a):,})', color='tab:green')
    ax.set_title(f'{title}  med pass={a.median():.0f}  filt={b.median():.0f}')
    ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(OUT_DIR / 'lengths.png'); plt.show()

In [ ]:
# Token-budget table for setting max_completion_length
qs = [0.5, 0.9, 0.95, 0.99, 1.0]
pd.DataFrame({
    'thinking_words'    : df.loc[df.pass_gate, 'thinking_words'].quantile(qs),
    'answer_words'      : df.loc[df.pass_gate, 'answer_words'].quantile(qs),
    'num_output_tokens' : df.loc[df.pass_gate, 'num_output_tokens'].quantile(qs),
}).round(0)

## 9. Diagnostics: position bias, truncation, grounding presence

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# best_idx histogram (sample-position bias)
vc = df.best_idx.value_counts().sort_index()
axes[0].bar(vc.index, vc.values)
axes[0].set_title('best_idx distribution (uniform ≈ no position bias)')
axes[0].set_xlabel('sample_idx'); axes[0].set_ylabel('count')

# finish_reason
fr = df.finish_reason.value_counts()
axes[1].bar(fr.index.astype(str), fr.values)
axes[1].set_title('winner finish_reason')
for i, v in enumerate(fr.values):
    axes[1].text(i, v, str(v), ha='center', va='bottom', fontsize=8)

# Grounding-object count
go = df.grounding_objs.clip(upper=10).value_counts().sort_index()
axes[2].bar(go.index, go.values)
axes[2].set_title('<objN> tags in winning trace')
axes[2].set_xlabel('# objects'); axes[2].set_ylabel('count')
plt.tight_layout(); plt.savefig(OUT_DIR / 'diagnostics.png'); plt.show()

print(f'\nfraction of passed winners with >=1 grounding obj: {(df.loc[df.pass_gate, "grounding_objs"]>0).mean():.3f}')

In [ ]:
# Score vs length scatter
sub = df.dropna(subset=['score_total','thinking_words']).sample(min(8000, len(df)), random_state=0)
fig, ax = plt.subplots(figsize=(7,4))
ax.scatter(sub.thinking_words, sub.score_total, s=3, alpha=0.25)
ax.set_xlabel('thinking words'); ax.set_ylabel('judge score_total')
ax.set_title('verbosity vs judge score')
ax.set_xlim(0, sub.thinking_words.quantile(0.995))
plt.tight_layout(); plt.savefig(OUT_DIR / 'length_vs_score.png'); plt.show()

In [ ]:
# Gate-pass rate per task_family (sorted bar)
fam_plot = fam.copy().sort_values('pass_rate', ascending=True)
fig, ax = plt.subplots(figsize=(8, max(3, 0.3*len(fam_plot))))
ax.barh(fam_plot.index, fam_plot.pass_rate)
for i, (n, p) in enumerate(zip(fam_plot.n, fam_plot.pass_rate)):
    ax.text(p, i, f'  {p*100:.0f}%  (n={n})', va='center', fontsize=8)
ax.set_xlim(0, 1); ax.set_xlabel('gate pass rate'); ax.set_title('D1 pass rate by task_family')
plt.tight_layout(); plt.savefig(OUT_DIR / 'pass_rate_by_family.png'); plt.show()

## 9d. Intrinsic reasoning-trace quality

Sections 6–9 score the winners with the *judge's* axes and check coverage/length. But the D1 gate
says nothing about properties of the reasoning **text itself** that matter for an SFT target. This
section computes those directly over each winning trace and surfaces traces that pass D1 yet are
still poor targets:

| characteristic | what it catches | metric |
|---|---|---|
| **language purity** | Qwen code-switching to Chinese mid-trace | CJK char fraction in `thinking`/`answer` |
| **degeneration** | looping / repeated phrasing | max 4-gram repetition, type-token ratio |
| **reasoning structure** | multi-step vs one-liner | sentence count, logical-connective count |
| **self-correction** | genuine deliberation vs straight-shot | reflection/backtracking-marker rate |
| **grounding utilization** | are `<objN>` tags actually used in reasoning, not just listed | `obj_used / obj_defined` |
| **answer derivation** | is the final answer reached in the trace, or tacked on | answer string ∈ `thinking` |

The closing cell rolls these into per-trace **red flags** and reports how many *gate-passed* winners
carry each one — the actionable view for trimming the SFT set.

In [ ]:
# === 9d. Intrinsic reasoning-trace quality features (winning B1 traces) ===
# These describe the trace TEXT itself — properties the D1 judge gate does not directly
# enforce, but which decide whether a winner is actually a good SFT target.

CJK_RE     = re.compile(r'[぀-ヿ㐀-䶿一-鿿가-힯ｦ-ﾟ]')
SENT_RE    = re.compile(r'[.!?。！？]+(?:\s|$)')
CONNECT_RE = re.compile(
    r'\b(because|therefore|thus|hence|since|so that|as a result|which means|'
    r'this (?:means|implies|suggests|indicates|tells us)|consequently|given that)\b',
    re.IGNORECASE)
# REFLECT_RE is already defined in 5c; reuse it if present, else define here for standalone runs.
try:
    REFLECT_RE
except NameError:
    REFLECT_RE = re.compile(
        r'\b(wait|but actually|hmm|let me (?:check|reconsider|re-?evaluate|verify|think again)|'
        r'on second thought|reconsider|actually,|on closer|however|alternatively|'
        r'could (?:also|instead)|that would mean|let me re-?check)\b',
        re.IGNORECASE)

def cjk_frac(s):
    s = s or ''
    return len(CJK_RE.findall(s)) / max(1, len(s))

def max_ngram_rep(s, n=4):
    toks = WORD_RE.findall((s or '').lower())
    if len(toks) <= n:
        return 0.0
    grams = [' '.join(toks[i:i+n]) for i in range(len(toks) - n + 1)]
    return Counter(grams).most_common(1)[0][1] / len(grams)

def ttr(s):
    toks = WORD_RE.findall((s or '').lower())
    return len(set(toks)) / max(1, len(toks))

def _norm(s):
    return re.sub(r'\s+', ' ', (s or '').strip().lower())

q_feats = []
for r in df.itertuples(index=False):
    w = winners.get((r.id, r.best_idx), {})
    think = w.get('thinking', '') or ''
    ans   = w.get('answer', '') or ''
    grd   = w.get('grounding', '') or ''

    defined = set(OBJ_RE.findall(grd))
    used    = defined & set(OBJ_RE.findall(think))
    a = _norm(ans)

    q_feats.append(dict(
        id              = r.id,
        sent_count      = len(SENT_RE.findall(think)),
        connective_n    = len(CONNECT_RE.findall(think)),
        reflect_n       = len(REFLECT_RE.findall(think)),
        has_reflection  = bool(REFLECT_RE.search(think)),
        think_cjk_frac  = cjk_frac(think),
        ans_cjk_frac    = cjk_frac(ans),
        rep_4gram       = max_ngram_rep(think, 4),
        ttr             = ttr(think),
        obj_defined     = len(defined),
        obj_used        = len(used),
        ground_util     = (len(used) / len(defined)) if defined else np.nan,
        answer_in_think = bool(a) and (a in _norm(think)),
    ))

qdf = pd.DataFrame(q_feats)
df = df.merge(qdf, on='id', how='left')

# Trace-level red flags (quality risks that survive the D1 gate)
df['flag_cjk']         = df.think_cjk_frac.fillna(0) > 0.005          # >0.5% CJK chars -> code-switch
df['flag_degenerate']  = df.rep_4gram.fillna(0) > 0.10               # repeated 4-gram loop
df['flag_no_reflect']  = ~df.has_reflection.fillna(False)            # one-shot, no self-check
df['flag_no_ground']   = (df.obj_defined > 0) & (df.obj_used == 0)   # grounding ignored in reasoning
df['flag_ans_floating']= ~df.answer_in_think.fillna(False)           # final answer not derived in trace

print('reasoning-trace quality features computed over', len(qdf), 'winners')
df[['sent_count', 'connective_n', 'reflect_n', 'think_cjk_frac',
    'rep_4gram', 'ttr', 'ground_util']].describe().round(3)

In [ ]:
# --- Language purity: CJK / code-switch leakage in winning traces ---
# Qwen3-VL sometimes switches to Chinese mid-trace. For an English SFT set this is a hard defect
# the judge axes do not catch. Even a small fraction of CJK chars usually means a switched clause.
any_cjk_think = (df.think_cjk_frac > 0).mean()
any_cjk_ans   = (df.ans_cjk_frac   > 0).mean()
print(f'traces with ANY CJK in thinking         : {any_cjk_think*100:.2f}%')
print(f'answers with ANY CJK                    : {any_cjk_ans*100:.2f}%')
print(f'traces flagged code-switch (>0.5% chars): {df.flag_cjk.mean()*100:.2f}%')
print(f'among GATE-PASSED winners, code-switch   : {df.loc[df.pass_gate, "flag_cjk"].mean()*100:.2f}%')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
nz = df.think_cjk_frac[df.think_cjk_frac > 0]
if len(nz):
    axes[0].hist(np.log10(nz.clip(lower=1e-5)), bins=40, edgecolor='black')
axes[0].set_xlabel('log10(CJK char fraction)  [traces with any CJK]')
axes[0].set_ylabel('# traces'); axes[0].set_title(f'CJK leakage in thinking (n={len(nz):,} have some)')

fam_cjk = df.groupby('task_family').flag_cjk.mean().sort_values()
axes[1].barh(fam_cjk.index, fam_cjk.values*100)
axes[1].set_xlabel('% traces flagged code-switch')
axes[1].set_title('Code-switch rate by task_family')
plt.tight_layout(); plt.savefig(OUT_DIR / 'language_purity.png'); plt.show()

In [ ]:
# --- Degeneration: n-gram repetition & lexical diversity ---
# Looping / repeated phrasing is a classic teacher-sampling failure and a terrible SFT target.
print(f'mean max-4gram repetition : {df.rep_4gram.mean():.3f}  (flagged >0.10: {df.flag_degenerate.mean()*100:.2f}%)')
print(f'mean type-token ratio     : {df.ttr.mean():.3f}')
print(f'degenerate AND gate-passed: {df.loc[df.pass_gate, "flag_degenerate"].mean()*100:.2f}%')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(df.rep_4gram.dropna(), bins=50, edgecolor='black')
axes[0].axvline(0.10, color='r', ls='--', label='flag threshold')
axes[0].set_xlabel('max 4-gram repetition'); axes[0].set_ylabel('# traces')
axes[0].set_title('Repetition (loop detector)'); axes[0].legend()

axes[1].hist(df.ttr.dropna(), bins=40, edgecolor='black')
axes[1].set_xlabel('type-token ratio'); axes[1].set_title('Lexical diversity')

# long + repetitive traces are the worst targets
sub = df.dropna(subset=['rep_4gram', 'thinking_words']).sample(min(8000, len(df)), random_state=0)
axes[2].scatter(sub.thinking_words, sub.rep_4gram, s=3, alpha=0.2)
axes[2].set_xlim(0, sub.thinking_words.quantile(.99))
axes[2].set_xlabel('thinking words'); axes[2].set_ylabel('max 4-gram rep')
axes[2].set_title('Repetition vs length')
plt.tight_layout(); plt.savefig(OUT_DIR / 'degeneration.png'); plt.show()

In [ ]:
# --- Reasoning structure & self-correction vs judge reasoning_quality ---
print(f'mean sentences/trace   : {df.sent_count.mean():.1f}')
print(f'mean connectives/trace : {df.connective_n.mean():.1f}')
print(f'traces with reflection : {df.has_reflection.mean()*100:.1f}%')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(df.sent_count.clip(upper=df.sent_count.quantile(.99)), bins=40, edgecolor='black')
axes[0].set_xlabel('# sentences'); axes[0].set_ylabel('# traces'); axes[0].set_title('Reasoning step count')

# reflection rate by judge reasoning_quality score — does self-correction track perceived quality?
rq = df.dropna(subset=['reason']).groupby('reason').has_reflection.mean()
axes[1].bar(rq.index, rq.values*100)
axes[1].set_xlabel('judge reasoning_quality'); axes[1].set_ylabel('% with reflection')
axes[1].set_title('Self-correction vs judge reason-quality')

# connective density by correctness
for lab, gsub in [('correct', df[df.ans_correct == 1]), ('wrong', df[df.ans_correct == 0])]:
    axes[2].hist(gsub.connective_n.clip(upper=20), bins=20, alpha=0.5, label=lab, density=True)
axes[2].set_xlabel('# logical connectives'); axes[2].set_ylabel('density')
axes[2].set_title('Connective density by correctness'); axes[2].legend()
plt.tight_layout(); plt.savefig(OUT_DIR / 'reasoning_structure.png'); plt.show()

In [ ]:
# --- Grounding utilization: are <objN> tags actually used in the reasoning? ---
# The judge scores visual_grounding, but a trace can name objects in the grounding block
# and never reference them while reasoning. This measures that follow-through.
has_obj = df[df.obj_defined > 0]
print(f'winners with >=1 grounded obj      : {len(has_obj):,} ({len(has_obj)/len(df)*100:.1f}%)')
print(f'  of those, reasoning references 0  : {(has_obj.obj_used == 0).mean()*100:.1f}%  (grounding ignored)')
print(f'  mean utilization (used/defined)   : {has_obj.ground_util.mean():.2f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(has_obj.ground_util.dropna(), bins=30, edgecolor='black')
axes[0].set_xlabel('fraction of grounded objs referenced in thinking')
axes[0].set_ylabel('# traces'); axes[0].set_title('Grounding utilization')

gu = has_obj.dropna(subset=['ground']).groupby('ground').ground_util.mean()
axes[1].bar(gu.index, gu.values)
axes[1].set_xlabel('judge visual_grounding score'); axes[1].set_ylabel('mean utilization')
axes[1].set_title('Utilization vs judge visual_grounding')
plt.tight_layout(); plt.savefig(OUT_DIR / 'grounding_utilization.png'); plt.show()

In [ ]:
# --- Red-flag summary: latent quality issues AMONG gate-passed winners ---
# These traces all clear D1, but may still be poor SFT targets. This is the actionable table.
flags = ['flag_cjk', 'flag_degenerate', 'flag_no_reflect', 'flag_no_ground', 'flag_ans_floating']
labels = {'flag_cjk': 'CJK code-switch', 'flag_degenerate': 'degenerate/looping',
          'flag_no_reflect': 'no self-correction', 'flag_no_ground': 'grounding ignored',
          'flag_ans_floating': 'answer not derived in trace'}
g = df[df.pass_gate]
tbl = pd.DataFrame({
    'all_winners_%': [df[f].mean()*100 for f in flags],
    'gate_passed_%': [g[f].mean()*100 for f in flags],
    'gate_passed_n': [int(g[f].sum()) for f in flags],
}, index=[labels[f] for f in flags]).round(2)
display(tbl)

clean = g[~g[flags].any(axis=1)]
print(f'gate-passed winners with NO red flag: {len(clean):,} / {len(g):,} ({len(clean)/max(1,len(g))*100:.1f}%)')

# Boilerplate / templated openings (SFT diversity check)
def opening(rid, bidx, k=8):
    t = winners.get((rid, bidx), {}).get('thinking') or ''
    return ' '.join(WORD_RE.findall(t.lower())[:k])

openers = Counter(opening(r.id, r.best_idx) for r in g.itertuples(index=False) if r.thinking_words > 0)
print('\nTop-15 opening 8-grams among gate-passed traces (high counts => templated reasoning):')
for phrase, n in openers.most_common(15):
    print(f'  {n:5d}  {phrase}')

## 9e. Reasoning-trace quality: B1 teacher vs C polish, head-to-head

Section 5c compared B1 and C on *correctness* and *length*. Here we run the **same intrinsic
trace-quality features from 9d on both** the teacher's winning trace (`thinking`) and the
text-only polish (`c2_think`), so we can see how the polish reshapes the reasoning itself:

- **structure** — sentences / logical connectives (does C preserve multi-step reasoning?)
- **self-correction** — reflection-marker rate (text-only re-derivation tends to be more linear)
- **code-switch** — any-CJK rate (which model leaks more non-English into the trace?)
- **degeneration / diversity** — 4-gram repetition, type-token ratio

The `C/B1` ratio column is the headline: <1 means the polish compresses that dimension.

In [ ]:
# --- B1 (teacher) vs C (text-only polish): intrinsic reasoning-trace quality, side by side ---
# Reuses phase_c (loaded in 5c) and the feature fns from 9d. Same functions applied to both traces.
def _feats(text):
    return dict(
        words   = len(WORD_RE.findall(text or '')),
        sents   = len(SENT_RE.findall(text or '')),
        connect = len(CONNECT_RE.findall(text or '')),
        reflect = int(bool(REFLECT_RE.search(text or ''))),
        cjk     = cjk_frac(text),
        rep4    = max_ngram_rep(text, 4),
        ttr     = ttr(text),
    )

cmp_rows = []
for rid, c in phase_c.items():
    if rid not in df_idx.index:
        continue
    w = winners.get((rid, df_idx.loc[rid, 'best_idx']))
    if not w:
        continue
    b  = _feats(w.get('thinking', ''))
    cc = _feats(c.get('c2_think', ''))
    cmp_rows.append({**{f'B1_{k}': v for k, v in b.items()},
                     **{f'C_{k}':  v for k, v in cc.items()}})
qc = pd.DataFrame(cmp_rows)
print(f'B1<->C intrinsic-quality pairs: {len(qc):,}\n')

metrics = ['words', 'sents', 'connect', 'reflect', 'cjk', 'rep4', 'ttr']
qsumm = pd.DataFrame({
    'B1_teacher': [qc[f'B1_{m}'].mean() for m in metrics],
    'C_polish'  : [qc[f'C_{m}'].mean()  for m in metrics],
}, index=metrics)
qsumm['C/B1'] = (qsumm.C_polish / qsumm.B1_teacher.replace(0, np.nan)).round(2)
display(qsumm.round(3))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
m = (qc.B1_words > 0) & (qc.C_words > 0)
hi = float(np.percentile(np.r_[qc.B1_sents[m], qc.C_sents[m]], 99))
bins = np.linspace(0, hi, 40)
axes[0].hist(qc.B1_sents[m], bins=bins, alpha=.55, label='B1 teacher', color='tab:blue')
axes[0].hist(qc.C_sents[m],  bins=bins, alpha=.55, label='C polish',  color='tab:orange')
axes[0].set_xlabel('# sentences'); axes[0].set_title('Reasoning steps: B1 vs C'); axes[0].legend()

axes[1].bar(['B1', 'C'], [qc.B1_reflect.mean()*100, qc.C_reflect.mean()*100], color=['tab:blue', 'tab:orange'])
axes[1].set_ylabel('% with self-correction'); axes[1].set_title('Reflection rate: B1 vs C')

axes[2].bar(['B1', 'C'], [(qc.B1_cjk > 0).mean()*100, (qc.C_cjk > 0).mean()*100], color=['tab:blue', 'tab:orange'])
axes[2].set_ylabel('% with any CJK'); axes[2].set_title('Code-switch rate: B1 vs C')
plt.tight_layout(); plt.savefig(OUT_DIR / 'b1_vs_c_trace_quality.png'); plt.show()

## 9f. Cognitive reasoning behaviors: B1 teacher vs C polish

Beyond the surface stats in 9d/9e, these are the canonical **reasoning behaviors** from the
self-improving-reasoner literature — the moves that separate genuine deliberation from linear
answer-writing. We detect each in both the teacher's winning trace (`thinking`) and the
text-only DeepSeek polish (`c2_think`):

- **backtracking** — abandons a line after noticing it's wrong ("wait, that's not right … let me redo")
- **verification** — explicitly checks an intermediate/final result ("let me double-check … does this match?")
- **subgoal_setting** — decomposes into ordered intermediate steps ("first … then … step 2")
- **branching** — explores alternatives / case analysis ("alternatively … case 1 / case 2 … either…or")
- **backward_chaining** — reasons from the goal back to the givens ("to find X we first need Y, which requires …")
- **deduction** *(added for contrast)* — forward chaining: givens ⇒ conclusion ("…, therefore / thus / hence …")

Detected by hand-tuned lexical cues (**recall-oriented, heuristic** — read as a *relative*
B1-vs-C signal, not absolute ground truth). Per behavior we report **presence rate** (% of
traces exhibiting it) and **mean occurrences/trace**, with the **C/B1** ratio as the headline
(<1 ⇒ the polish strips that behavior out).

> **Calibration note.** On these traces `subgoal_setting`/`backward_chaining` come out very low
> — that is **genuine, not a detector miss** (validated against raw token freq: `in order to` 0%,
> `requires` 0%, vs `thus` 53% / `therefore` 51% / `since` 41%). The spatial questions are *atomic*
> ("291 < 744, so the worker is further left"), so the traces are dominated by **forward-chaining
> deduction** rather than goal decomposition or working backward. The `deduction` row makes that explicit.

In [ ]:
# --- Cognitive reasoning behaviors: B1 teacher winning trace vs C text-only polish ---
# Lexical detectors for the canonical self-improving-reasoner behaviors (backtracking,
# verification, subgoal-setting, branching, backward-chaining). Heuristic & recall-oriented:
# use as a RELATIVE B1-vs-C signal, not absolute ground truth. Reuses phase_c / df_idx / winners
# from 5c. Each value counts NON-overlapping cue matches in the trace.
COG_BEHAVIORS = {
    'backtracking': re.compile(
        r"\b(?:wait|hold on|scratch that|never ?mind|on second thought|back ?track(?:ing)?|"
        r"let me (?:redo|reconsider|rethink|re-?examine|re-?evaluate|start over|revise)|"
        r"that'?s (?:wrong|incorrect|not right|not correct)|i made a mistake|"
        r"that doesn'?t (?:work|seem right|match|add up)|but (?:that'?s|wait)|"
        r"going back|correction:|let me correct|hmm)\b", re.I),
    'verification': re.compile(
        r"\b(?:let me (?:verify|check|double-?check|confirm|make sure)|"
        r"to (?:verify|confirm|be sure|double-?check)|sanity[ -]?check|double check|"
        r"plug(?:ging)? (?:it |this |that )?back|does (?:this|that) (?:make sense|check out|hold|match)|"
        r"verify that|this (?:checks out|confirms|is consistent)|(?:is|are) consistent with|"
        r"confirm(?:s|ing)? (?:that|the)|as a check|let me confirm|to make sure)\b", re.I),
    'subgoal_setting': re.compile(
        r"\b(?:first,|firstly,|second,|secondly,|next,? (?:i|we|let|check|compare)|"
        r"then (?:i|we) (?:need|will|can|compare|check|look)|step \d|step one|step two|"
        r"let me (?:start|begin|first|determine|identify|establish|figure)|"
        r"i need to (?:first|determine|find|compare|figure|identify|establish)|"
        r"i'?ll (?:first|start by|need to)|break (?:this|it|the problem) (?:down|into)|"
        r"sub-?(?:goal|problem|task|step)|the (?:plan|steps?) (?:is|are)|"
        r"let'?s (?:start|begin|figure)|begin by|to (?:answer|solve) this,? (?:i|we|let))\b", re.I),
    'branching': re.compile(
        r"\b(?:alternativ(?:e|ely)|another (?:way|approach|option|possibility)|on the other hand|"
        r"case \d|case one|case two|in (?:one|the first|the second) case|option [ab12]\b|"
        r"two (?:possibilities|options|ways|cases)|we could (?:also|instead)|or (?:we|i) could|"
        r"consider (?:both|two|the other)|either way|but if|if (?:instead|however))\b", re.I),
    'backward_chaining': re.compile(
        r"\b(?:in order to|work(?:ing)? backwards?|(?:that|which) requires|"
        r"requires (?:knowing|finding|computing|first|the)|"
        r"to (?:determine|find|know|get|compute|answer)[^.,;]{1,55}?"
        r"(?:i need|we need|i must|we must|requires|need to know|need the|i first|we first)|"
        r"need to (?:know|find)[^.,;]{1,45}? (?:first|before)|so (?:i|we) first need|"
        r"depends on (?:knowing|the|whether))\b", re.I),
    'deduction': re.compile(
        r"\b(?:therefore|thus|hence|consequently|it follows that|as a result|"
        r"this (?:means|implies)|which (?:means|implies))\b", re.I),
}

def cog_counts(text):
    t = text or ''
    return {name: len(rx.findall(t)) for name, rx in COG_BEHAVIORS.items()}

rows = []
for rid, c in phase_c.items():
    if rid not in df_idx.index:
        continue
    w = winners.get((rid, df_idx.loc[rid, 'best_idx']))
    if not w:
        continue
    b  = cog_counts(w.get('thinking', ''))
    cc = cog_counts(c.get('c2_think', ''))
    rows.append({**{f'B1_{k}': v for k, v in b.items()},
                 **{f'C_{k}':  v for k, v in cc.items()}})
cog = pd.DataFrame(rows)
print(f'B1<->C cognitive-behavior pairs: {len(cog):,}\n')

behaviors = list(COG_BEHAVIORS)
summ = pd.DataFrame({
    'B1_present_%': [(cog[f'B1_{b}'] > 0).mean()*100 for b in behaviors],
    'C_present_%' : [(cog[f'C_{b}']  > 0).mean()*100 for b in behaviors],
    'B1_mean_n'   : [cog[f'B1_{b}'].mean() for b in behaviors],
    'C_mean_n'    : [cog[f'C_{b}'].mean()  for b in behaviors],
}, index=behaviors)
summ['present_C/B1'] = (summ['C_present_%'] / summ['B1_present_%'].replace(0, np.nan)).round(2)
display(summ.round(2))

b1p = summ['B1_present_%'].values; cp = summ['C_present_%'].values
b1n = summ['B1_mean_n'].values;    cn = summ['C_mean_n'].values
y = np.arange(len(behaviors)); h = 0.38
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
axes[0].barh(y - h/2, b1p, h, label='B1 teacher', color='tab:blue')
axes[0].barh(y + h/2, cp,  h, label='C polish',  color='tab:orange')
for i in range(len(behaviors)):
    axes[0].text(b1p[i], i - h/2, f' {b1p[i]:.0f}%', va='center', fontsize=8)
    axes[0].text(cp[i],  i + h/2, f' {cp[i]:.0f}%',  va='center', fontsize=8)
axes[0].set_yticks(y); axes[0].set_yticklabels(behaviors)
axes[0].set_xlabel('% of traces with behavior'); axes[0].legend(fontsize=8)
axes[0].set_title('Cognitive-behavior presence: B1 vs C')

axes[1].barh(y - h/2, b1n, h, label='B1 teacher', color='tab:blue')
axes[1].barh(y + h/2, cn,  h, label='C polish',  color='tab:orange')
axes[1].set_yticks(y); axes[1].set_yticklabels(behaviors)
axes[1].set_xlabel('mean occurrences / trace'); axes[1].legend(fontsize=8)
axes[1].set_title('Cognitive-behavior density: B1 vs C')
plt.tight_layout(); plt.savefig(OUT_DIR / 'b1_vs_c_cognitive_behaviors.png'); plt.show()

# Reasoning richness: how many DISTINCT behaviors co-occur in one trace
cog['B1_nb'] = sum((cog[f'B1_{b}'] > 0).astype(int) for b in behaviors)
cog['C_nb']  = sum((cog[f'C_{b}']  > 0).astype(int) for b in behaviors)
print(f'\nmean distinct behaviors/trace : B1={cog.B1_nb.mean():.2f}  C={cog.C_nb.mean():.2f}  (of {len(behaviors)})')
print(f'traces with >=3 behaviors     : B1={(cog.B1_nb>=3).mean()*100:.1f}%  C={(cog.C_nb>=3).mean()*100:.1f}%')
print(f'traces with 0 behaviors       : B1={(cog.B1_nb==0).mean()*100:.1f}%  C={(cog.C_nb==0).mean()*100:.1f}%')


## 10. Qualitative inspector — top / borderline / failed

In [ ]:
def show_example(row, header):
    qa = qa_by_id[row.id]
    w  = winners.get((row.id, row.best_idx), {})
    j  = judge_by_id[row.id]
    md  = f'### {header} — `{row.id}`  total={row.score_total}  '
    md += f'(hallu={row.hallu} ground={row.ground} reason={row.reason} correct={row.ans_correct})\n\n'
    md += f'**task_family:** {row.task_family} / {row.template_type}  \n'
    md += f'**gold:** {qa.get("gt_answer")}  \n'
    md += f'**question:** {qa.get("prompt", "").split("Question:")[-1].split("Options:")[0].strip()[:400]}\n\n'
    md += f'**scene_description (judge):** {(j.get("scene_description") or "")[:400]}…\n\n'
    md += f'**grounding block:**\n```\n{(w.get("grounding") or "")[:500]}\n```\n'
    md += f'**thinking:** {(w.get("thinking") or "")[:800]}…\n\n'
    md += f'**answer:** `{(w.get("answer") or "").strip()[:300]}`\n'
    display(Markdown(md))
    try:
        img = Image.open(qa['image_path'])
        fig, ax = plt.subplots(figsize=(5,3)); ax.imshow(img); ax.axis('off'); plt.show()
    except Exception as e:
        print(f'(image load failed: {e})')

rng = random.Random(0)
passed = df[df.pass_gate]
failed = df[~df.pass_gate & df.score_total.notna()]

top  = passed.nlargest(5, 'score_total')
bord = passed[passed.score_total.between(passed.score_total.quantile(0.4), passed.score_total.quantile(0.6))].sample(min(5, 200), random_state=0)
fail = failed.nsmallest(5, 'score_total')

for _, r in top.iterrows():  show_example(r, '🟢 TOP')
for _, r in bord.iterrows(): show_example(r, '🟡 BORDERLINE (passed, median score)')
for _, r in fail.iterrows(): show_example(r, '🔴 FAILED (low score)')

## 11. Export gate-pass id set for the Stage D formatter

In [ ]:
summary = {
    'judge_file'      : str(JUDGE_FILE),
    'cot_file'        : str(COT_FILE),
    'qa_file'         : str(QA_FILE),
    'gate'            : GATE,
    'n_total'         : int(N),
    'n_passed'        : int(df.pass_gate.sum()),
    'pass_rate'       : float(df.pass_gate.mean()),
    'per_task_family' : fam.to_dict(orient='index'),
    'token_budget_p95': float(df.loc[df.pass_gate, 'num_output_tokens'].quantile(0.95)) if df.pass_gate.any() else None,
}
(OUT_DIR / 'stage_d_audit_summary.json').write_text(json.dumps(summary, indent=2, default=str))

passed_ids = df.loc[df.pass_gate, ['id','best_idx']].to_dict(orient='records')
(OUT_DIR / 'stage_d_passed_ids.jsonl').write_text(''.join(json.dumps(r)+'\n' for r in passed_ids))

df.to_parquet(OUT_DIR / 'audit_df.parquet', index=False)
print('wrote:')
for p in OUT_DIR.iterdir():
    print(' ', p.name, f'{p.stat().st_size/1e6:.2f} MB')